# Analysis de Datos

### Imports

In [ ]:
import os
import requests
import pandas as pd
from sqlalchemy import create_engine, text

# paths
CSV_PATH = "/home/jovyan/datos/sensores.csv"
SQLITE_DIR = Path("/home/jovyan/work/data")
SQLITE_DIR.mkdir(parents=True, exist_ok=True)

PG_URI = "postgresql+psycopg2://smartcampus:smartpampus_dev@db:5432/smartcampus"
SQLITE_URI = f"sqlite:///{SQLITE_DIR / 'smartcampus.db'}"

### read Data, check structure and prepare for storage

In [ ]:
# 1. Dimensions check (192 rows, 5 columns)
assert len(df) == 192, f"Expected 192 rows, got: {len(df)}"
assert set(df.columns) == {"edificio", "fecha_hora", "temperatura", "humedad", "co2_ppm"}, "Column schema mismatch."

# 2. Null values and duplicates check
assert df.isna().sum().sum() == 0, "Dataset contains null values."
assert df.duplicated().sum() == 0, "Dataset contains duplicate records."

# 3. Physical boundaries validation
assert df["temperatura"].between(-20, 60).all(), "Temperature values out of physical bounds."
assert df["humedad"].between(0, 100).all(), "Relative humidity values out of range [0, 100]."
assert (df["co2_ppm"] >= 0).all(), "Negative CO2 readings detected."

# Explicit type enforcement for persistence
df["edificio"] = df["edificio"].astype(str)
df["temperatura"] = df["temperatura"].astype(float)
df["humedad"] = df["humedad"].astype(float)
df["co2_ppm"] = df["co2_ppm"].astype(int)

print("Data quality checks completed successfully.")

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=["fecha_hora"])

In [ ]:
pg_engine = create_engine(PG_URI)

# Idempotent table truncation within an atomic transaction
with pg_engine.begin() as conn:
    conn.execute(text("DELETE FROM lecturas;"))

# Append data
df.to_sql("lecturas", con=pg_engine, if_exists="append", index=False)

# Verify loaded row count
with pg_engine.connect() as conn:
    pg_count = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

assert pg_count == 192, f"Incorrect row count in PostgreSQL: {pg_count} != 192"
print(f"Records successfully loaded into PostgreSQL: {pg_count}")

In [ ]:
# A) Engine-side SQL aggregation: Average temperature (ANSI SQL)
query_temp = text("""
    SELECT 
        edificio, 
        ROUND(AVG(temperatura), 2) AS temp_media
    FROM lecturas 
    GROUP BY edificio 
    ORDER BY edificio;
""")
df_temp_pg = pd.read_sql(query_temp, con=pg_engine)
print("Average temperature per building (Engine-side SQL):")
print(df_temp_pg)

# B) In-memory DataFrame aggregation: Average CO2 via full table extraction
df_lecturas_pg = pd.read_sql_table("lecturas", con=pg_engine)
df_co2_pg = (
    df_lecturas_pg.groupby("edificio", as_index=False)["co2_ppm"]
    .mean()
    .rename(columns={"co2_ppm": "co2_medio"})
)
df_co2_pg["co2_medio"] = df_co2_pg["co2_medio"].round(1)

# Identification of building with maximum CO2
idx_max_co2_pg = df_co2_pg["co2_medio"].idxmax()
top_co2_pg_edificio = df_co2_pg.loc[idx_max_co2_pg, "edificio"]
top_co2_pg_valor = df_co2_pg.loc[idx_max_co2_pg, "co2_medio"]

print("\nAverage CO2 per building (In-memory pandas):")
print(df_co2_pg)
print(f"\nBuilding with highest mean CO2: Building {top_co2_pg_edificio} ({top_co2_pg_valor} ppm)")

### SQL Lite

In [ ]:
sqlite_engine = create_engine(SQLITE_URI)

# Clean up table in SQLite
with sqlite_engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS lecturas;"))

# Append data
df.to_sql("lecturas", con=sqlite_engine, if_exists="append", index=False)

# Verify row count
with sqlite_engine.connect() as conn:
    sqlite_count = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

assert sqlite_count == 192, f"Incorrect row count in SQLite: {sqlite_count} != 192"
print(f"Records successfully loaded into SQLite: {sqlite_count}")

### Unified function

In [ ]:
def cargar_y_resumir(engine, data: pd.DataFrame) -> dict:
    """
    Vendor-agnostic data pipeline function.
    Operates without internal branching based on database dialect.
    """
    # 1. Clear table to ensure idempotent execution
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM lecturas;"))
    
    # 2. Persistence
    data.to_sql("lecturas", con=engine, if_exists="append", index=False)
    
    # 3. Row count extraction via SQLAlchemy Core
    with engine.connect() as conn:
        total_rows = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()
    
    # 4. SQL Aggregation: Average temperature (ANSI SQL)
    sql_query = text("""
        SELECT 
            edificio, 
            ROUND(AVG(temperatura), 2) AS temp_media 
        FROM lecturas 
        GROUP BY edificio 
        ORDER BY edificio;
    """)
    resumen_temp = pd.read_sql(sql_query, con=engine)
    
    # 5. pandas Aggregation: Average CO2 via full table extraction
    df_completo = pd.read_sql_table("lecturas", con=engine)
    resumen_co2 = (
        df_completo.groupby("edificio", as_index=False)["co2_ppm"]
        .mean()
        .rename(columns={"co2_ppm": "co2_medio"})
    )
    resumen_co2["co2_medio"] = resumen_co2["co2_medio"].round(1)
    
    max_co2_idx = resumen_co2["co2_medio"].idxmax()
    edificio_max_co2 = resumen_co2.loc[max_co2_idx, "edificio"]
    valor_max_co2 = resumen_co2.loc[max_co2_idx, "co2_medio"]
    
    return {
        "total_registros": total_rows,
        "temperatura_media": resumen_temp,
        "co2_medio": resumen_co2,
        "edificio_max_co2": (edificio_max_co2, valor_max_co2)
    }

# Execute unified pipeline on both database engines
res_pg = cargar_y_resumir(pg_engine, df)
res_sqlite = cargar_y_resumir(sqlite_engine, df)

print("Unified pipeline executed successfully across both engines.")

### strict scalar assertions

In [ ]:
assert res_pg["total_registros"] == res_sqlite["total_registros"] == 192, "Row count mismatch between engines."
assert res_pg["edificio_max_co2"] == res_sqlite["edificio_max_co2"] == ("D", 1349.3), "Max CO2 mismatch between engines."

# DataFrame comparison for CO2 results
pd.testing.assert_frame_equal(res_pg["co2_medio"], res_sqlite["co2_medio"])

print("Comparative Temperature Results:")
print("--- PostgreSQL ---")
print(res_pg["temperatura_media"])
print("--- SQLite ---")
print(res_sqlite["temperatura_media"])

print("\nComparative CO2 Results:")
print("--- PostgreSQL ---")
print(res_pg["co2_medio"])
print("--- SQLite ---")
print(res_sqlite["co2_medio"])